# 01 · Algonauts data preparation

This notebook handles acquisition, permanent Google Drive backup, and session restore of the Algonauts 2025 data required by the BEATs-to-fMRI project.

It manages all three required categories in one complete acquisition:
- one Friends / Schaefer-1000 fMRI `.h5` file for each configured subject;
- all available video parts (`a`, `b`, …) for Season 2 Episodes 1–5;
- one subject-specific Schaefer-1000 atlas NIfTI volume for each configured subject, stored in `fmri/<subject>/atlas/`.

The DataLad/git-annex setup is handled before acquisition. The installer now uses non-interactive privilege handling (`--sudo ok`), tries the official release and DataLad package sources, and falls back to the system `apt` package where available. If installation still fails, the notebook displays the captured installer diagnostics and stops before downloading data.

**Run order:** mount Drive → install/verify DataLad and git-annex → optionally fetch all three categories → back up to Drive → restore to local disk → verify fMRI, stimuli, and atlases.

For a complete initial acquisition, set `RUN_DATA_DOWNLOAD = True` in Section 2. For an existing backup that only needs atlas files, leave the full-download switch off and use Section 2.1.


## 1. Mount Google Drive and locate the project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib.util
import os
import re
import subprocess
import sys

# Current project folder in Google Drive. Change only if you moved it.
PROJECT_DIR = Path('/content/drive/MyDrive/BEATs_algonauts_project')
if not (PROJECT_DIR / 'config.py').exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}. Update PROJECT_DIR above.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ['BEATS_BRAIN_IN_COLAB'] = '1'
os.environ['BEATS_BRAIN_DRIVE_ROOT'] = str(PROJECT_DIR)
os.environ['BEATS_BRAIN_WORK_DIR'] = str(PROJECT_DIR / 'outputs')

import config
import get_algonauts_data as algonauts

print('Project:', PROJECT_DIR)
print('Permanent Algonauts backup:', config.DRIVE_DATA)
print('Local working data:', config.ALGONAUTS_ROOT)
print('Configured subjects:', config.SUBJECTS)
print('Atlas location:', f'fmri/<subject>/{config.ATLAS_DIRNAME}/')
print('Atlas filename pattern:', config.ATLAS_GLOB)


Mounted at /content/drive
Project: /content/drive/MyDrive/BEATs_algonauts_project
Permanent Algonauts backup: /content/drive/MyDrive/algonauts_2025
Local working data: /content/data
Configured subjects: ['sub-01', 'sub-02', 'sub-03', 'sub-05']
Atlas location: fmri/<subject>/atlas/
Atlas filename pattern: *.nii*


## 2. Download and back up the dataset

Set `RUN_DATA_DOWNLOAD = True` for a complete acquisition. The fetch includes **fMRI + all selected video parts + the four subject-specific atlas volumes**. The files are first fetched into the local DataLad dataset under `/content`, then copied into the permanent Google Drive backup at `config.DRIVE_DATA` while preserving their original directory structure.

The tool preflight runs once before the fetch. It upgrades/installs DataLad and `datalad-installer`, attempts `git-annex` installation non-interactively from the official release source, then the DataLad package archive, and finally the system package manager where available. The output of failed installer attempts is retained for diagnosis.

After a successful full backup, set `RUN_DATA_DOWNLOAD = False` before routinely rerunning the notebook. DataLad will not re-transfer files already present in the current local dataset, but a fresh Colab runtime needs to fetch the data again from the remote before backing up unless you restore from Drive.


In [2]:
RUN_DATA_DOWNLOAD = True  # Complete initial acquisition: fMRI + all selected videos + all four atlases.
RUN_ATLAS_ONLY_DOWNLOAD = False  # Set True only to extend an existing fMRI/video Drive backup.

if RUN_DATA_DOWNLOAD and RUN_ATLAS_ONLY_DOWNLOAD:
    raise ValueError('Choose either the full download or atlas-only download, not both.')

if RUN_DATA_DOWNLOAD or RUN_ATLAS_ONLY_DOWNLOAD:
    print('Preparing DataLad and git-annex before any dataset fetch...')
    algonauts.ensure_tools()
    print('DataLad/git-annex preflight passed; acquisition can proceed.')

if RUN_DATA_DOWNLOAD:
    repo, selection = algonauts.fetch_all(
        parent_dir='/content',
        skip_tools=True,  # ensure_tools() was run explicitly above.
        dry_run=False,
        categories=algonauts.CATEGORIES,  # ('fmri', 'clips', 'atlas')
    )
    print('\nSelected files for the complete acquisition:')
    print(algonauts.describe(selection, repo, categories=algonauts.CATEGORIES))

    present, total = algonauts.fetch_report(selection, categories=algonauts.CATEGORIES)
    if present != total:
        raise RuntimeError(f'Only {present}/{total} selected files are present after DataLad fetch.')

    if config.DRIVE_DATA is None:
        raise RuntimeError('DRIVE_DATA is not configured; check ALGONAUTS_DRIVE_DATA in config.py / environment.')

    copied, skipped, missing = algonauts.backup_to_drive(
        selection, repo, config.DRIVE_DATA, categories=algonauts.CATEGORIES
    )
    if missing:
        raise RuntimeError(f'{missing} selected file(s) could not be copied to Drive.')
    print(f'\nFull Drive backup complete: copied={copied}, already present={skipped}, missing={missing}')
else:
    print('Skipping full dataset download.')


Preparing DataLad and git-annex before any dataset fetch...
$ /usr/bin/python3 -m pip install --quiet --upgrade datalad datalad-installer
$ datalad --version
It is highly recommended to configure Git before using DataLad. Set both 'user.name' and 'user.email' configuration variables.
datalad 1.7.3
$ /usr/local/bin/datalad-installer --sudo ok --log-level INFO git-annex -m datalad/git-annex:release
2026-11-02T19:11:26+0000 [INFO    ] datalad_installer: Writing environment modifications to /tmp/dl-env-75lorl0l.sh
2026-11-02T19:11:26+0000 [INFO    ] datalad_installer: Installing git-annex via datalad/git-annex:release
2026-11-02T19:11:26+0000 [INFO    ] datalad_installer: Version: None
2026-11-02T19:11:26+0000 [INFO    ] datalad_installer: Downloading https://github.com/con/git-annex/releases/download/10.20261026/git-annex-standalone_10.20261026-1.ndall%2B1_amd64.deb
2026-11-02T19:11:28+0000 [INFO    ] datalad_installer: Running: sudo dpkg -i /tmp/tmped09la6s/git-annex-standalone_10.202610

### 2.1. Optional: add only the subject-specific atlases to an existing backup

Use this only when your Drive backup already contains the fMRI and video files. Set `RUN_DATA_DOWNLOAD = False` and `RUN_ATLAS_ONLY_DOWNLOAD = True` in Section 2.

This branch installs/verifies the tools if needed, reuses or installs the local DataLad metadata clone, selects exactly one NIfTI file from `fmri/<subject>/atlas/` for each configured subject, fetches only those atlas paths, and copies only those files to the existing `config.DRIVE_DATA` backup. It does not request fMRI or video paths from DataLad.

The default pattern is `*.nii*`, supporting `.nii` and `.nii.gz`. If a subject's atlas directory contains more than one NIfTI file, the downloader stops and lists the candidates rather than guessing.


In [3]:
if RUN_ATLAS_ONLY_DOWNLOAD:
    if config.DRIVE_DATA is None:
        raise RuntimeError('DRIVE_DATA is not configured; set ALGONAUTS_DRIVE_DATA in config.py / environment.')

    repo, selection = algonauts.fetch_all(
        parent_dir='/content',
        skip_tools=True,  # ensure_tools() was run once in Section 2.
        dry_run=False,
        categories=('atlas',),
    )
    print('\nAtlas files selected:')
    print(algonauts.describe(selection, repo, categories=('atlas',)))

    present, total = algonauts.fetch_report(selection, categories=('atlas',))
    expected = len(config.SUBJECTS)
    if total != expected or present != expected:
        raise RuntimeError(f'Expected {expected} downloaded atlases; found {present}/{total}.')

    copied, skipped, missing = algonauts.backup_to_drive(
        selection, repo, config.DRIVE_DATA, categories=('atlas',)
    )
    if missing:
        raise RuntimeError(f'{missing} atlas file(s) were missing during Drive backup.')
    print(f'\nAtlas-only backup complete: copied={copied}, already present={skipped}, missing={missing}')
else:
    print('Skipping atlas-only download.')


Skipping atlas-only download.


## 3. Restore the complete permanent backup to local disk

This copies the contents of `config.DRIVE_DATA` into `config.ALGONAUTS_ROOT`. It is safe to rerun: files with the same size are skipped. The restore includes any atlas files newly added in Section 2.1.


In [4]:
if config.DRIVE_DATA is None:
    raise RuntimeError('DRIVE_DATA is not configured. Check config.py / ALGONAUTS_DRIVE_DATA.')

copied, skipped = algonauts.restore_from_drive(config.DRIVE_DATA, config.ALGONAUTS_ROOT)
print(f'Restored dataset files: copied={copied}, already present={skipped}')


Restored dataset files: copied=18, already present=0


## 4. Verify the fMRI and video inputs required by the encoding pipeline

In [5]:
missing_fmri = []
for subject in config.SUBJECTS:
    hits = sorted((config.FMRI_DIR / subject / 'func').glob(config.FMRI_GLOB))
    print(f'{subject}: {len(hits)} matching fMRI file(s)')
    if len(hits) != 1:
        missing_fmri.append((subject, [p.name for p in hits]))

stimulus_dir = config.STIM_DIR / f's{config.SEASON}'
required_episodes = set(config.TRAIN_EPS) | {config.TEST_EP}
episode_alt = '|'.join(map(str, sorted(required_episodes)))
clip_pattern = re.compile(
    rf's0?-?{config.SEASON}e0?({episode_alt})[a-z]\.mkv$', re.IGNORECASE
)
clips = sorted(p for p in stimulus_dir.glob('*.mkv') if clip_pattern.search(p.name))
print(f'Video clips found: {len(clips)}')
for clip in clips:
    print(' -', clip.name)

found_episodes = {
    int(match.group(1))
    for path in clips
    if (match := re.search(r'e0?(\d+)[a-z]\.mkv$', path.name, re.IGNORECASE))
}
missing_episodes = sorted(required_episodes - found_episodes)
if missing_fmri or missing_episodes:
    raise FileNotFoundError(
        f'Encoding input verification failed. fMRI issues: {missing_fmri}; '
        f'missing video episodes: {missing_episodes}; video directory: {stimulus_dir}'
    )
print('fMRI and stimulus inputs are present.')


sub-01: 1 matching fMRI file(s)
sub-02: 1 matching fMRI file(s)
sub-03: 1 matching fMRI file(s)
sub-05: 1 matching fMRI file(s)
Video clips found: 10
 - friends_s02e01a.mkv
 - friends_s02e01b.mkv
 - friends_s02e02a.mkv
 - friends_s02e02b.mkv
 - friends_s02e03a.mkv
 - friends_s02e03b.mkv
 - friends_s02e04a.mkv
 - friends_s02e04b.mkv
 - friends_s02e05a.mkv
 - friends_s02e05b.mkv
fMRI and stimulus inputs are present.


## 5. Verify the subject-specific Schaefer-1000 atlas volumes

The Algonauts documentation states that each atlas is a 3D array of shape `(97, 115, 97)`, with parcel labels from 1 to 1,000 and zero outside the brain. The NIfTI affine must be retained because it maps voxel coordinates into world coordinates.

This validation checks that exactly one atlas is present for each subject, that the volume dimensions match the documentation, and that the labels are integer-valued and within the expected range. It does not assume that every subject must contain all 1,000 labels.


In [6]:
import numpy as np

# Install nibabel in this Colab session only when it is not already available.
if importlib.util.find_spec('nibabel') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel'], check=True)

from data import load_atlas

atlas_paths = {}
for subject in config.SUBJECTS:
    atlas_volume, atlas_affine, atlas_path = load_atlas(subject)
    atlas_paths[subject] = atlas_path
    labels = np.unique(atlas_volume)
    positive_labels = labels[labels > 0]
    print(
        f'{subject}: file={atlas_path.name}, shape={atlas_volume.shape}, '
        f'positive labels={positive_labels.size}, '
        f'label range={positive_labels.min()}–{positive_labels.max()}'
    )
    print('  affine:')
    print(atlas_affine)

print('\nAll subject atlases passed validation. Their NIfTI affines are available for later spatial mapping.')


sub-01: file=sub-01_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz, shape=(97, 115, 97), positive labels=1000, label range=1–1000
  affine:
[[   2.     0.    -0.   -96.5]
 [   0.     2.    -0.  -132.5]
 [   0.     0.     2.   -78.5]
 [   0.     0.     0.     1. ]]
sub-02: file=sub-02_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz, shape=(97, 115, 97), positive labels=1000, label range=1–1000
  affine:
[[   2.     0.    -0.   -96.5]
 [   0.     2.    -0.  -132.5]
 [   0.     0.     2.   -78.5]
 [   0.     0.     0.     1. ]]
sub-03: file=sub-03_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz, shape=(97, 115, 97), positive labels=1000, label range=1–1000
  affine:
[[   2.     0.    -0.   -96.5]
 [   0.     2.    -0.  -132.5]
 [   0.     0.     2.   -78.5]
 [   0.     0.     0.     1. ]]
sub-05: file=sub-05_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-